# GDBN v10: Weekly Model with Weekly Inputs (Point-in-Time)

Notebooks 09 and 10 predicted 5-day returns but still made a **new prediction every day** (overlapping weeks). Here everything is **weekly**:

- **One row per week** (Monday to Friday; the week is labeled by its last trading day).
- **Inputs:** last week's totals for each stock, plus four new inputs.
- **Target:** next week's excess return (vs the universe average that week).
- **Portfolio:** re-ranked **once a week** at Friday's close. Buy the top 20%, short the bottom 20%, and hold for the week.

### Inputs (for each stock, all known at Friday's close)

| Input | Meaning | New? |
|---|---|---|
| `ret_w` | last week's excess return (compounded) | |
| `ret_m` | the past 4 weeks' excess return | |
| `sent_w` | last week's news sentiment, **weighted by stories** (a 10-story day counts 10× a 1-story day) | |
| `attn_w` | log(1 + total stories last week) | |
| `vol_w` | **weekly volatility:** log of the typical size of last week's daily moves (direction ignored) | new |
| `surprise` | **news surprise:** log(1 + stories last week) − log(1 + the stock's average weekly stories over the previous 26 weeks) | new |
| `sent_chg` | **sentiment change:** last week's sentiment − the previous 4 weeks' sentiment (both weighted by stories) | new |
| `earn_w` | **earnings week:** 1 if the stock's earnings became public last week | new |
| ETFs | last week's return of SPY, USO, GLD, DBC, TLT | |

**Timing (no look-ahead):** Compustat gives the earnings report **date** but not the time, and many reports come after the close. So a report counts from the **next trading day**, the same rule as notebook 11. A report after Friday's close counts toward the following week.

### Two GDBNs and four simple benchmarks

| Model | Inputs |
|---|---|
| **GDBN basic** | `ret_w`, `ret_m`, `sent_w`, `attn_w`, ETFs (like notebook 10, but weekly) |
| **GDBN + new inputs** | the basic inputs + `vol_w`, `surprise`, `sent_chg`, `earn_w` |
| Benchmark: weekly reversal | buy last week's losers, short last week's winners (−`ret_w`) |
| Benchmark: news surprise | rank by `surprise` |
| Benchmark: sentiment change | rank by `sent_chg` |
| Benchmark: low volatility | rank by −`vol_w` (calm stocks long, volatile stocks short) |

A benchmark with a **negative** IC means the opposite direction worked.

### Setup
- **Universe:** the same point-in-time universe (5 largest per sector, re-selected each quarter). A week uses the universe of the latest quarter that started on or before the week's first day.
- **Walk-forward:** the network is re-learned at the start of each quarter, on up to **260 weeks (5 years)** before it, with at least 150 weeks. The first years have less history (the data starts in 2003).
- **λ:** one shared λ for each GDBN, chosen by the best average weekly rank IC over 2006–2020. The grid stops at 0.005 (the daily notebooks go down to 0.001): with about 420 inputs and only about 260 training weeks, a smaller λ would let the lasso memorize the training weeks. **Validation:** 2021–2022. **Test years are not used.**
- **Costs:** 5 bps per unit traded, as before. Sharpe ratios are annualized with √52.

Takes about 7 minutes (the model with the new inputs is the slow part).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import lasso_path

import pit_common as pc
from gdbn_utils import RESULTS

WINDOW_W, MIN_W, LOOK_W = 260, 150, 5    # training window in weeks (max, min), extra weeks for the 4-week return
MIN_OBS_W = 104                          # a stock needs 2 years of weekly returns in the window
ALPHAS_W = pc.ALPHAS[pc.ALPHAS >= 0.005]  # λ grid: smaller λ would let ~420 inputs memorize ~260 weeks
FUND = pc.BUILT.parent / "fundamentals"

D = pc.load()                            # everything after 2022 is blanked out (test years locked)

# ----------------------------------------------------------------------------
# Weeks: each trading day -> its week (Mon-Fri), labeled by the week's last trading day
# ----------------------------------------------------------------------------
days = D.days[D.days <= pc.EVAL_END]
period = days.to_period("W-FRI")
last_day = pd.Series(days, index=days).groupby(period).transform("max")
WK = pd.Series(last_day.values, index=days)                  # day -> week label
FIRST = pd.Series(days, index=days).groupby(WK.values).min()  # week label -> first trading day
WEEKS = pd.DatetimeIndex(FIRST.index)

def weekly_sum(df):
    return df.loc[days].groupby(WK.values).sum(min_count=1)

RET = D.RET.loc[days]
WRET = np.expm1(weekly_sum(np.log1p(RET)))                                 # compounded weekly return
WVOL = np.log(np.sqrt((RET ** 2).groupby(WK.values).mean()) + 1e-4)
WVOL = WVOL.where(RET.notna().groupby(WK.values).sum() >= 2)               # needs at least 2 days
NEWS = np.expm1(D.ATTN.loc[days])                                          # stories per day
WN = weekly_sum(NEWS)                                                      # stories per week
WSN = weekly_sum(D.SENT.loc[days] * NEWS)                                  # sum of sentiment x stories
WSENT = WSN / WN.where(WN > 0)                                             # story-weighted sentiment (NaN = no news)
SURP = np.log1p(WN) - np.log1p(WN.shift(1).rolling(26, min_periods=8).mean())
SENT_M = WSN.shift(1).rolling(4, min_periods=1).sum() / WN.shift(1).rolling(4, min_periods=1).sum().where(lambda x: x > 0)
SCHG = WSENT - SENT_M
ETFW = np.expm1(weekly_sum(np.log1p(D.ETF.loc[days])))

# Earnings weeks: report date (rdq) from Compustat -> next trading day -> week; gvkey -> permno via the CCM link on that date
fq = pd.read_csv(FUND / "fundq.csv", usecols=["gvkey", "datadate", "rdq"], parse_dates=["datadate", "rdq"], dtype={"gvkey": str})
fq = fq.dropna(subset=["rdq"]).drop_duplicates(["gvkey", "rdq"])
idx = days.searchsorted(fq["rdq"].values, side="right")
fq = fq[idx < len(days)].assign(usable=days[idx[idx < len(days)]])
ccm = pd.read_csv(pc.BUILT.parent / "ccm_links.csv", parse_dates=["linkdt", "linkenddt"], dtype={"gvkey": str})
ev = fq.merge(ccm[["gvkey", "permno", "linkdt", "linkenddt"]], on="gvkey")
ev = ev[(ev["usable"] >= ev["linkdt"]) & (ev["usable"] <= ev["linkenddt"].fillna(pd.Timestamp("2100-01-01")))]
ev["week"] = WK.reindex(ev["usable"]).values
ev = ev.dropna(subset=["week", "permno"]).drop_duplicates(["permno", "week"])
EARN = pd.DataFrame(0.0, index=WEEKS, columns=WRET.columns)
for p, w in zip(ev["permno"].astype(int), ev["week"]):
    if p in EARN.columns:
        EARN.loc[w, p] = 1.0

print(f"{len(WEEKS):,} weeks ({WEEKS[0].date()} to {WEEKS[-1].date()}); {len(D.rebal)} quarters; "
      f"earnings events matched: {int(EARN.values.sum()):,}")

## 1. A look at the weekly data

One stock's last 8 weeks before 2019 (Apple), to check the inputs make sense. (The universe-relative excess return is computed inside the walk-forward, so the raw weekly return is shown here.)

In [ ]:
aapl = 14593
show = pd.DataFrame({"return": WRET[aapl], "vol_w": WVOL[aapl], "stories": WN[aapl], "sent_w": WSENT[aapl],
                     "surprise": SURP[aapl], "sent_chg": SCHG[aapl], "earn_w": EARN[aapl]}).loc["2018-11-01":"2018-12-31"]
print(show.round(3).to_string())
print("\nShare of stock-weeks with any news:", f"{(WN > 0).sum().sum() / WN.notna().sum().sum():.0%}")

## 2. Walk-forward

The same steps as `pit_common.walk_forward` (z-scores from the training window, one lasso per stock, the whole λ grid at once), with weeks instead of days. The inputs are shifted by one week, so each week is predicted only from the weeks before it.

In [ ]:
def basic_features(span, U, EX):
    return pd.concat([EX.add_suffix("|ret_w"), EX.rolling(4, min_periods=2).sum().add_suffix("|ret_m"),
                      WSENT.reindex(index=span, columns=U).add_suffix("|sent_w"),
                      np.log1p(WN.reindex(index=span, columns=U)).add_suffix("|attn_w"),
                      ETFW.reindex(span).add_suffix("|etf")], axis=1)


def new_features(span, U, EX):
    return pd.concat([basic_features(span, U, EX),
                      WVOL.reindex(index=span, columns=U).add_suffix("|vol_w"),
                      SURP.reindex(index=span, columns=U).add_suffix("|surprise"),
                      SCHG.reindex(index=span, columns=U).add_suffix("|sent_chg"),
                      EARN.reindex(index=span, columns=U).add_suffix("|earn_w")], axis=1)


def walk_forward_weekly(features, label):
    out, t0 = [], time.time()
    for j, r in enumerate(D.rebal):
        nxt = D.rebal[j + 1] if j + 1 < len(D.rebal) else pd.Timestamp(pc.EVAL_END) + pd.Timedelta(days=1)
        hold = WEEKS[(FIRST >= r).values & (FIRST < nxt).values]               # weeks that start in this quarter
        if len(hold) == 0:
            continue
        pos = WEEKS.get_loc(hold[0])
        n_win = min(WINDOW_W, pos - LOOK_W)
        if n_win < MIN_W:
            continue
        win = WEEKS[pos - n_win: pos]                                        # every training week ends before the quarter
        span = WEEKS[pos - n_win - LOOK_W: pos + len(hold)]
        U = D.U[r]
        R = WRET.reindex(index=span, columns=U)
        EX = R.sub(R.mean(axis=1), axis=0)
        X = features(span, U, EX).shift(1)                                    # information through last week
        Xw, Xh = X.loc[win], X.loc[hold]
        mu, sd = Xw.mean(), Xw.std().replace(0, np.nan)
        Xwz, Xhz = ((Xw - mu) / sd).fillna(0).values, ((Xh - mu) / sd).fillna(0).values
        xm = Xwz.mean(axis=0)
        S = np.full((len(ALPHAS_W), len(hold), len(U)), np.nan)
        W = np.zeros((len(U), X.shape[1], len(ALPHAS_W)))
        for i, p in enumerate(U):
            y = EX.loc[win, p].values
            ok = ~np.isnan(y)
            if ok.sum() < MIN_OBS_W:
                continue
            ys = y[ok].std()
            _, coefs, _ = lasso_path(Xwz[ok] - xm, (y[ok] - y[ok].mean()) / ys, alphas=ALPHAS_W)
            W[i] = coefs
            S[:, :, i] = ((Xhz - xm) @ coefs).T * ys
        out.append({"r": r, "hold": hold, "U": U, "S": S, "W": W, "cols": list(X.columns), "n_win": n_win})
        if (j + 1) % 20 == 0:
            print(f"  {label}: {j + 1} of {len(D.rebal)} quarters ({time.time() - t0:.0f}s)")
    print(f"{label}: done in {time.time() - t0:.0f}s; first week predicted {out[0]['hold'][0].date()} "
          f"(window {out[0]['n_win']} weeks), inputs per stock-quarter: {len(out[-1]['cols'])}")
    return out


res = {"GDBN basic": walk_forward_weekly(basic_features, "GDBN basic"),
       "GDBN + new inputs": walk_forward_weekly(new_features, "GDBN + new inputs")}

## 3. Choose λ (best weekly rank IC on 2006–2020)

In [ ]:
def fwd_excess(score):
    # Each stock's return in the predicted week, minus the average of that week's universe
    r = WRET.reindex(index=score.index, columns=score.columns).where(score.notna())
    return r.sub(r.mean(axis=1), axis=0)


chosen = {}
fig, ax = plt.subplots(figsize=(9, 3.5))
for name, rs in res.items():
    target = fwd_excess(pc.scores(rs, 0))
    ics = np.array([pc.rank_ic(pc.scores(rs, k).loc[:pc.TUNE_END], target.loc[:pc.TUNE_END]).mean() for k in range(len(ALPHAS_W))])
    k = int(np.nanargmax(ics))
    arrows = np.mean([(q["W"][:, :, k] != 0).sum() for q in rs])
    chosen[name] = k
    ax.semilogx(ALPHAS_W, ics, "o-", ms=3, label=f"{name} (λ = {ALPHAS_W[k]:.4f}, ≈{arrows:.0f} arrows per quarter)")
    print(f"{name}: λ = {ALPHAS_W[k]:.4f}, 2006–2020 weekly IC = {ics[k]:.4f}, arrows per quarter ≈ {arrows:.0f}")
ax.axhline(0, color="black", lw=0.8); ax.invert_xaxis(); ax.legend(fontsize=8)
ax.set_xlabel("λ  (left: few arrows, right: many arrows)"); ax.set_ylabel("mean weekly rank IC, 2006–2020")
ax.set_title("Weekly rank IC along the λ grid"); plt.tight_layout(); plt.show()

## 4. Results

**IC:** the average weekly rank correlation between the score and the week's excess return. **bps/week:** the long–short return per week before costs. **Breakeven bps:** profit per unit traded; above about 5 bps the strategy survives realistic costs. Newey–West t-statistics use 4 lags.

For comparison, notebook 10's best daily model (slow inputs, held 5 days) had a 2006–2020 IC of 0.021 (against 5-day returns), net Sharpe 0.05, and breakeven 5.6 bps; in 2021–2022, net Sharpe 0.56 and breakeven 11.1 bps.

In [ ]:
def summary_w(score, start, end):
    s = score.loc[start:end]
    w = pc.daily_portfolio(s)                                      # top / bottom 20% of each week's universe
    r = WRET.reindex(index=w.index, columns=w.columns).fillna(0.0)
    gross = (w * r).sum(axis=1)
    turnover = w.diff().abs().sum(axis=1)
    turnover.iloc[0] = w.iloc[0].abs().sum()
    net = gross - turnover * pc.COST_BPS / 1e4
    ic = pc.rank_ic(s, fwd_excess(s))
    return pd.Series({"IC": ic.mean(), "IC t": pc.nw_t(ic, 4), "bps/week": gross.mean() * 1e4, "t": pc.nw_t(gross, 4),
                      "Sharpe gross": gross.mean() / gross.std() * np.sqrt(52), "Sharpe net": net.mean() / net.std() * np.sqrt(52),
                      "turnover/week": turnover.mean(), "breakeven bps": gross.mean() / turnover.mean() * 1e4, "weeks": len(ic)})


signals = {name: pc.scores(rs, chosen[name]) for name, rs in res.items()}
inuni = signals["GDBN basic"].notna()
prev = lambda df: df.shift(1).reindex(index=inuni.index, columns=inuni.columns).where(inuni)   # last week's value
signals.update({"Benchmark: weekly reversal": -prev(WRET), "Benchmark: news surprise": prev(SURP),
                "Benchmark: sentiment change": prev(SCHG), "Benchmark: low volatility": -prev(WVOL)})

tables = {}
for period, (a, b) in pc.PERIODS.items():
    tab = pd.DataFrame({n: summary_w(s, a, b) for n, s in signals.items()}).T
    spy = ETFW.loc[inuni.index, "SPY"].loc[a:b]
    tab.loc["S&P 500 (SPY, buy & hold)", ["bps/week", "t", "Sharpe gross", "Sharpe net"]] = [
        spy.mean() * 1e4, pc.nw_t(spy, 4), spy.mean() / spy.std() * np.sqrt(52), spy.mean() / spy.std() * np.sqrt(52)]
    tables[period] = tab
    print(f"===== {period} =====")
    print(tab.round(3).to_string(), "\n")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, s in signals.items():
    if name.startswith("Benchmark: news") or name.startswith("Benchmark: sentiment"):
        continue
    w = pc.daily_portfolio(s)
    g = (w * WRET.reindex(index=w.index, columns=w.columns).fillna(0)).sum(axis=1)
    axes[0].plot((1 + g).cumprod(), label=name)
    axes[1].plot(pc.rank_ic(s, fwd_excess(s)).rolling(26).mean(), label=name)
for ax in axes:
    ax.axvline(pd.Timestamp("2021-01-01"), color="gray", ls="--", lw=1); ax.legend(fontsize=8)
axes[0].set_yscale("log"); axes[0].set_title("Weekly long–short, growth of $1 (before costs; dashed = validation)")
axes[1].axhline(0, color="black", lw=0.8); axes[1].set_title("Weekly rank IC, 26-week rolling average")
plt.tight_layout(); plt.show()

## 5. What does the network use?

Share of arrows by input type (averaged over all quarters, at the chosen λ), and the average sign of each input's arrow **into its own stock** (for example, does a stock's own news surprise predict its own next week up or down?).

In [ ]:
def kind(c):
    return c.split("|")[1]

for name, rs in res.items():
    k = chosen[name]
    counts = pd.Series(0.0, index=sorted({kind(c) for c in rs[-1]["cols"]}))
    own = {kd: [] for kd in counts.index}
    for q in rs:
        nz = q["W"][:, :, k] != 0
        for j, c in enumerate(q["cols"]):
            counts[kind(c)] += nz[:, j].sum()
        for i, p in enumerate(q["U"]):
            for j, c in enumerate(q["cols"]):
                if c.startswith(f"{p}|") and nz[i, j]:
                    own[kind(c)].append(q["W"][i, j, k])
    tab = pd.DataFrame({"share of arrows": counts / counts.sum(),
                        "own-stock arrows": {kd: len(v) for kd, v in own.items()},
                        "own-stock: share positive": {kd: np.mean(np.array(v) > 0) if v else np.nan for kd, v in own.items()}})
    print(f"===== {name} =====")
    print(tab.sort_values("share of arrows", ascending=False).round(3).to_string(), "\n")

**Reading it:**
- **GDBN + new inputs vs GDBN basic:** do the four new inputs add anything once the network can already see returns, sentiment, and attention?
- **GDBN vs weekly reversal:** if the GDBN does not beat this simple rule, its gain is mostly the known reversal effect (as in notebook 10).
- **Single-input benchmarks:** show whether each new input has any ranking power on its own, and in which direction.
- **Weekly vs daily (notebook 10):** compare the Sharpe ratios and the breakeven bps.

## 6. Save results

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
pd.concat(tables, names=["period", "model"]).to_csv(RESULTS / "pit_weekly_results.csv")
pd.Series({n: ALPHAS_W[k] for n, k in chosen.items()}, name="lambda").to_csv(RESULTS / "pit_weekly_lambda.csv")
print("Saved to", RESULTS)